## 1. Import and inspect data

In [1]:
import pandas as pd

df = pd.read_json("../data/raw/acs_raw.json", orient="records", dtype=str)

print("A look at the raw data...")
print()
df.info()
print()
df.describe()

A look at the raw data...

<class 'pandas.DataFrame'>
RangeIndex: 2327 entries, 0 to 2326
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   NAME         2327 non-null   str  
 1   B19013_001E  2327 non-null   str  
 2   state        2327 non-null   str  
 3   county       2327 non-null   str  
 4   tract        2327 non-null   str  
 5   borough      2327 non-null   str  
dtypes: str(6)
memory usage: 109.2 KB



,NAME,B19013_001E,state,county,tract,borough
count,2327,2327,2327,2327,2327,2327
unique,2327,2102,1,5,1530,5
top,Census Tract 1; New York County; New York,-666666666,36,047,003900,brooklyn
freq,1,135,2327,805,5,805


## 2. Rename & drop unnecessary columns

In [2]:
# rename columns
df = df.rename(columns={'B19013_001E' : 'income'}) # rename column for median household income
df.columns = df.columns.str.strip().str.lower()

# drop columns
df.drop(labels=["name", "state", "borough"], axis=1, inplace=True) # we only need census tract & county for geographic reference

print("Inspect row names")
print()
df.info()

Inspect row names

<class 'pandas.DataFrame'>
RangeIndex: 2327 entries, 0 to 2326
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   income  2327 non-null   str  
 1   county  2327 non-null   str  
 2   tract   2327 non-null   str  
dtypes: str(3)
memory usage: 54.7 KB


## 3. Fix data types

In [3]:
# safely convert income to int
df['income'] = pd.to_numeric(df['income'])
# don't convert numerical county and tract identifiers to numerics, want to preserve leading zeroes
print("Inspect new typing: ")
print(df.dtypes)

Inspect new typing: 
income    int64
county      str
tract       str
dtype: object


## 4. Standardize data entries

In [4]:
print("test for null and duplicate rows pre-cleaning: ")
print("Null counts: ")
print(df.isnull().sum())
print(f"Starting duplicate rows: {df.duplicated().sum()}")
print(f"Missing values in income rows: {(df['income'] == (-666666666)).sum()}")

# standardize data
df = df[df['income'] != -666666666] # drop rows with placeholder null val in income column

print()
print("Post cleaning:")
print(f"Remaining missing values in income rows: {(df['income'] == (-666666666)).sum()}")

test for null and duplicate rows pre-cleaning: 
Null counts: 
income    0
county    0
tract     0
dtype: int64
Starting duplicate rows: 0
Missing values in income rows: 135

Post cleaning:
Remaining missing values in income rows: 0


## 5. Final check

In [5]:
print("data shape: ")
print(df.shape)
print()

print("null counts:")
print(df.isnull().sum())
print()

print("data types:")
print(df.dtypes)
print()

print("first few entries:")
df.head(10)

data shape: 
(2192, 3)

null counts:
income    0
county    0
tract     0
dtype: int64

data types:
income    int64
county      str
tract       str
dtype: object

first few entries:


,income,county,tract
1,38657,061,000201
2,40182,061,000202
4,34473,061,000600
5,197058,061,000700
6,33953,061,000800
7,222237,061,000900
8,130417,061,001001
9,24350,061,001002
10,105938,061,001200
11,193984,061,001300


## 6. Export final data

In [ ]:
df.to_pickle('../data/clean/acs_cleaned.pkl') # pickle file preserves datatypes to we don't lose critical info
print(f'Exported {len(df)} rows as pkl')
df.to_csv('../data/clean/acs_cleaned.csv')
print(f'Exported {len(df)} rows as csv')

Exported 2192 rows
